In [1]:
import sys
sys.path.append("../..")

In [2]:
from syntax_tokenizer import SyntaxTokenizer
from model import ModelConfig, LlamaModel
from train import TrainerConfig, SimpleDataLoader, Trainer

In [3]:
with open("../../data/complete_shakespeare.txt") as f:
    text = f.read()

In [4]:
texts = text.split("\n\n\n\n")

In [5]:
tokenizer = SyntaxTokenizer()
tokenizer.train(texts)

153it [01:00,  2.54it/s]


In [6]:
tokenizer.vocab_size

12852

In [7]:
model_config = ModelConfig(
    vocab_size=tokenizer.vocab_size,
    d_model=576,
    d_head=64,
    d_mlp_proj=1536,
    n_layers=30,
    n_kv_heads=3,
    n_attn_heads=9,
    rms_norm_eps=1e-5,
    initializer_range=0.02,
    rope_theta=100000.0,
    padding_idx=tokenizer.data.pad_token_id
)

In [8]:
train_config = TrainerConfig(
    per_device_train_batch_size=8,
    max_seq_len=512,
    num_epochs=8,
    eval_interval_steps=25,
    learning_rate=4e-3,
    grad_clip_norm=1.0,
    val_size=0.05,
    log_dir="runs/shakespeare_style1",
    warmup_ratio=0.1
)

In [9]:
n = 100
size_patch = len(text) // 100
texts_equal = [text[i:i+size_patch] for i in range(n+1)]

In [10]:
model = LlamaModel(model_config)
dataloader = SimpleDataLoader(train_config, tokenizer, texts=texts_equal)
trainer = Trainer(train_config, model, tokenizer)

Total tokens                   | 1,344,512
Num Trainable Params           | 121,008,960
Train device                   | cuda, NVIDIA GeForce RTX 3090, N=1
Training precision             | torch.bfloat16
Flash Attention                | True
torch.compile()                | True
DistributedDataParallel        | False
Batch size                     | 4,096




In [11]:
trainer.train(dataloader)

Training steps                 | 2,496 
Step: 0, Training Loss: 9.60051, LR: 0.0002000, Tokens/sec: 385.87
Step: 1, Training Loss: 8.48383, LR: 0.0002153, Tokens/sec: 439.06
Step: 2, Training Loss: 8.07717, LR: 0.0002305, Tokens/sec: 107396.06
Step: 3, Training Loss: 7.67372, LR: 0.0002458, Tokens/sec: 115266.10
Running test generate with input: The world is



>>>[('DT', 0, 1), ('NN', 2, 1), ('VBZ', 1, 1), ('_SP', 1, 0), (',', 1, 0), ('_SP', 1, 0), ('_SP', 1, 0), (',', 1, 0), ('_SP', 1, 0), ('_SP', 1, 0), ('_SP', 1, 0), ('_SP', 1, 0), ('_SP', 1, 0), ('_SP', 1, 0), ('_SP', 1, 0), (',', 1, 0), ('_SP', 1, 0), ('_SP', 1, 0), (',', 1, 0), ('_SP', 1, 0), ('_SP', 1, 0), ('_SP', 1, 0), ('_SP', 1, 0), ('_SP', 1, 0), ('_SP', 1, 0), ('_SP', 1, 0), ('_SP', 1, 0), ('_SP', 1, 0), ('_SP', 1, 0), (',', 1, 0), ('_SP', 1, 0), ('_SP', 1, 0), (',', 1, 0), (',', 1, 0), ('_SP', 1, 0)]

>>>[('DT', 0, 1), ('NN', 2, 1), ('VBZ', 1, 1), ('_SP', 1, 0), ('_SP', 1, 0), ('_SP', 1, 0), (',', 1, 0), ('_SP', 1, 0), ('

In [12]:
input_text = """
ALL. Content, content.
""".strip()

input_ids = tokenizer([input_text], return_tensors="pt")['input_ids'].to(trainer.device)
idx = model.generate(input_ids, temperature=2, top_k=500, max_new_tokens=32)
print(tokenizer.batch_decode(idx)[0])

[('NNP', 7, 1), ('.', 1, 0), ('NNP', 8, 2), (',', 1, 0), ('NN', 3, 2), ('.', 1, 0), ('_SP', 4, 0), ('WRB', 1, 1), ('MD', 2, 1), ('RB', 1, 1), ('PRP', 1, 1), ('VB', 3, 1), ('IN', 1, 1), ('PRP$', 1, 1), ('NN', 4, 1), ('JJ', 5, 1), ('_SP', 1, 0), ('CC', 3, 2), ('NN', 9, 2), (',', 1, 0), ('NN', 7, 2), ('JJS', 5, 2), ('IN', 0, 1), ('CD', 4, 1), ('_SP', 7, 0), ('VBN', 4, 1), (',', 1, 0), ('_SP', 1, 0), ('WRB', 2, 1), ('IN', 1, 1), ('PRP$', 1, 1), ('NN', 9, 2), ('IN', 1, 1), ('NN', 3, 2), ('POS', 2, 0), ('VBD', 7, 5), ('JJ', 2, 1), ('NN', 10, 3)]
